# Calculator Model Training and SHAP/FFA Workflow

**Purpose:** Train calculator models and run SHAP + Formal Feature Attribution (FFA) analysis  
**Updated:** January 26, 2026  
**Hardware:** Optimized for EC2 instances  
**Cohorts:** Combined, CHD, Myocardio

## Overview

This notebook provides an interactive workflow for:

1. **Training Calculator Models** - Train CatBoost, XGBoost, and XGBoost RF models for each cohort
2. **SHAP + FFA Analysis** - Generate causal factors and dashboard data using SHAP values and XGBoost rule extraction
3. **Results Inspection** - View top causal factors, feature importance, and model performance

## Workflow Steps

- **Step 1:** Train models for selected cohort(s)
- **Step 2:** Run SHAP/FFA analysis to extract causal factors
- **Step 3:** Inspect results and export dashboard data

## Expected Runtime

- **Model Training (per cohort):** ~10-30 minutes depending on cohort size
- **SHAP/FFA Analysis (per cohort):** ~5-15 minutes
- **Total (all 3 cohorts):** ~1-2 hours on EC2


## 1. Setup and Configuration

Load required packages and configure paths.

In [ ]:
import sys
from pathlib import Path
import logging
import warnings
warnings.filterwarnings('ignore')

# Add project paths
PROJECT_ROOT = Path().resolve().parent.parent.parent
CALCULATOR_DIR = Path().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(CALCULATOR_DIR))

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("=" * 80)
print("PHTS Calculator Workflow")
print("=" * 80)
print(f"Project root: {PROJECT_ROOT}")
print(f"Calculator directory: {CALCULATOR_DIR}")
print("=" * 80)

In [ ]:
# Configuration
DEBUG_MODE = False  # Set to True for quick testing (fewer splits)

# Cohort selection - can be a single cohort or list of cohorts
COHORTS = ["Combined"]  # Options: "Combined", "CHD", "Myocardio"

# SHAP/FFA configuration
TOP_K = 10  # Number of top causal factors to extract
WEIGHT_CATBOOST = 0.6  # Weight for CatBoost importance
WEIGHT_XGBOOST = 0.4  # Weight for XGBoost importance

print(f"\nConfiguration:")
print(f"  DEBUG_MODE: {DEBUG_MODE}")
print(f"  Cohorts: {COHORTS}")
print(f"  Top K factors: {TOP_K}")
print(f"  CatBoost weight: {WEIGHT_CATBOOST}")
print(f"  XGBoost weight: {WEIGHT_XGBOOST}")

In [ ]:
# Check dependencies
print("\nChecking dependencies...")

try:
    import numpy as np
    import pandas as pd
    from catboost import CatBoostRegressor
    import xgboost as xgb
    import shap
    print("✓ All required packages are installed")
    print(f"  NumPy: {np.__version__}")
    print(f"  Pandas: {pd.__version__}")
    print(f"  XGBoost: {xgb.__version__}")
    print(f"  SHAP: {shap.__version__}")
except ImportError as e:
    print(f"✗ Missing dependency: {e}")
    print("  Please install: pip install numpy pandas catboost xgboost shap")

In [ ]:
# Check data availability
print("\nChecking data availability...")

data_file = PROJECT_ROOT / "graft-loss" / "data" / "phts_txpl_ml.sas7bdat"
if data_file.exists():
    size_mb = data_file.stat().st_size / (1024 * 1024)
    print(f"✓ Data file found: {data_file}")
    print(f"  Size: {size_mb:.2f} MB")
else:
    print(f"⚠ Data file not found: {data_file}")
    print("  You may need to download the data file first")

# Check calculator directory structure
outputs_dir = CALCULATOR_DIR / "outputs"
if outputs_dir.exists():
    print(f"✓ Outputs directory exists: {outputs_dir}")
else:
    print(f"✓ Creating outputs directory: {outputs_dir}")
    outputs_dir.mkdir(parents=True, exist_ok=True)

## 2. Train Calculator Models

Train CatBoost, XGBoost, and XGBoost RF models for each selected cohort.

In [ ]:
# Import training function
from train_python_models import train_models_for_cohort

print(f"\n{'=' * 80}")
print("Training Calculator Models")
print(f"{'=' * 80}")

# Train models for each cohort
for cohort in COHORTS:
    print(f"\nTraining {cohort} model...")
    print("-" * 80)
    try:
        train_models_for_cohort(cohort)
        print(f"\n✓ {cohort} model training complete!")
    except Exception as e:
        print(f"\n✗ Error training {cohort} model: {e}")
        logger.error(f"Error training {cohort} model", exc_info=True)

print(f"\n{'=' * 80}")
print("Model training complete!")
print(f"{'=' * 80}")

In [ ]:
# Check training results
import json

print("\nTraining Results Summary:")
print("-" * 80)

for cohort in COHORTS:
    best_model_file = CALCULATOR_DIR / "outputs" / "models" / cohort / "best_model.txt"
    if best_model_file.exists():
        print(f"\n{cohort} Cohort:")
        with open(best_model_file, 'r') as f:
            print(f.read())
    else:
        print(f"\n⚠ {cohort}: Best model file not found")

    # List model files
    models_dir = CALCULATOR_DIR / "outputs" / "models" / cohort
    if models_dir.exists():
        model_files = list(models_dir.glob("*.cbm")) + list(models_dir.glob("*.ubj"))
        if model_files:
            print(f"  Model files ({len(model_files)}):")
            for model_file in sorted(model_files):
                size_mb = model_file.stat().st_size / (1024 * 1024)
                print(f"    {model_file.name} ({size_mb:.2f} MB)")

## 3. Run SHAP + FFA Analysis

Generate SHAP values and extract causal factors using Formal Feature Attribution.

In [ ]:
# Run SHAP/FFA workflow for each cohort
import subprocess

print(f"\n{'=' * 80}")
print("Running SHAP + FFA Analysis")
print(f"{'=' * 80}")

for cohort in COHORTS:
    print(f"\nRunning SHAP/FFA for {cohort} cohort...")
    print("-" * 80)
    
    try:
        result = subprocess.run(
            [
                sys.executable,
                str(CALCULATOR_DIR / "run_shap_ffa_workflow.py"),
                "--cohort", cohort,
                "--top-k", str(TOP_K),
                "--weight-catboost", str(WEIGHT_CATBOOST),
                "--weight-xgboost", str(WEIGHT_XGBOOST)
            ],
            cwd=str(CALCULATOR_DIR),
            capture_output=False,  # Show output in real-time
            text=True
        )
        
        if result.returncode == 0:
            print(f"\n✓ {cohort} SHAP/FFA analysis complete!")
        else:
            print(f"\n⚠ {cohort} SHAP/FFA exited with code: {result.returncode}")
    except Exception as e:
        print(f"\n✗ Error running SHAP/FFA for {cohort}: {e}")
        logger.error(f"Error running SHAP/FFA for {cohort}", exc_info=True)

print(f"\n{'=' * 80}")
print("SHAP/FFA analysis complete!")
print(f"{'=' * 80}")

## 4. Inspect Results

View top causal factors, feature importance, and dashboard data.

In [ ]:
# Load and display dashboard data
import json
import pandas as pd

print("\n" + "=" * 80)
print("Results Summary")
print("=" * 80)

for cohort in COHORTS:
    dashboard_data_file = (
        CALCULATOR_DIR / "outputs" / "shap_ffa" / cohort / "dashboard_data.json"
    )
    
    if dashboard_data_file.exists():
        print(f"\n{cohort} Cohort - Top {TOP_K} Causal Factors:")
        print("-" * 80)
        
        with open(dashboard_data_file, 'r') as f:
            dashboard_data = json.load(f)
        
        top_factors = dashboard_data.get('top_causal_factors', [])[:TOP_K]
        
        if top_factors:
            for idx, factor in enumerate(top_factors, 1):
                importance = factor.get('causal_responsibility', 
                                     factor.get('importance', 
                                               factor.get('combined_importance_norm', 0)))
                print(f"{idx:2d}. {factor['feature']:40s} "
                      f"(Importance: {importance:.4f})")
        else:
            print("  (No causal factors available)")
        
        # Display summary statistics
        if 'summary' in dashboard_data:
            print(f"\n  Summary Statistics:")
            summary = dashboard_data['summary']
            for key, value in summary.items():
                print(f"    {key}: {value}")
    else:
        print(f"\n⚠ {cohort}: Dashboard data not found")
        print(f"  Expected: {dashboard_data_file}")
        print("  Run SHAP/FFA analysis first (Section 3)")

In [ ]:
# Load and display feature importance
print("\n" + "=" * 80)
print("Feature Importance Rankings")
print("=" * 80)

for cohort in COHORTS:
    importance_files = list(
        (CALCULATOR_DIR / "outputs" / "models" / cohort).glob("importance_*.csv")
    )
    
    if importance_files:
        print(f"\n{cohort} Cohort - Feature Importance:")
        print("-" * 80)
        
        for imp_file in sorted(importance_files):
            model_name = imp_file.stem.replace(f"importance_{cohort}_", "")
            print(f"\n  {model_name}:")
            df = pd.read_csv(imp_file)
            print(f"    Total features: {len(df)}")
            print(f"    Top 5 features:")
            top5 = df.nlargest(5, 'importance')
            for idx, row in top5.iterrows():
                print(f"      {row['feature']:40s} {row['importance']:.4f}")
    else:
        print(f"\n⚠ {cohort}: No feature importance files found")
        print("  Train models first (Section 2)")

## 5. Visualizations (Optional)

Create visualizations of results.

In [ ]:
# Plot top causal factors (if matplotlib is available)
try:
    import matplotlib.pyplot as plt
    
    for cohort in COHORTS:
        dashboard_data_file = (
            CALCULATOR_DIR / "outputs" / "shap_ffa" / cohort / "dashboard_data.json"
        )
        
        if dashboard_data_file.exists():
            with open(dashboard_data_file, 'r') as f:
                dashboard_data = json.load(f)
            
            top_factors = dashboard_data.get('top_causal_factors', [])[:TOP_K]
            
            if top_factors:
                # Extract data for plotting
                features = [f['feature'] for f in top_factors]
                importance = [f.get('causal_responsibility', 
                                  f.get('importance', 
                                       f.get('combined_importance_norm', 0))) 
                            for f in top_factors]
                
                # Create plot
                plt.figure(figsize=(10, max(6, len(features) * 0.4)))
                plt.barh(range(len(features)), importance)
                plt.yticks(range(len(features)), features)
                plt.xlabel('Causal Responsibility / Importance')
                plt.title(f'Top {TOP_K} Causal Factors - {cohort} Cohort')
                plt.gca().invert_yaxis()  # Top factor at top
                plt.tight_layout()
                
                # Save plot
                plot_file = CALCULATOR_DIR / "outputs" / "shap_ffa" / cohort / f"top_{TOP_K}_factors.png"
                plt.savefig(plot_file, dpi=150, bbox_inches='tight')
                print(f"\n✓ Saved plot: {plot_file}")
                
                plt.show()
            
except ImportError:
    print("\n⚠ Matplotlib not available. Skipping visualizations.")
    print("  Install with: pip install matplotlib")

## 6. Export Summary

Create a summary JSON file with all results.

In [ ]:
# Create workflow summary
from datetime import datetime

summary = {
    "workflow": "Calculator Model Training + SHAP/FFA Analysis",
    "timestamp": datetime.now().isoformat(),
    "configuration": {
        "cohorts": COHORTS,
        "top_k": TOP_K,
        "weight_catboost": WEIGHT_CATBOOST,
        "weight_xgboost": WEIGHT_XGBOOST,
        "debug_mode": DEBUG_MODE
    },
    "cohorts": {}
}

for cohort in COHORTS:
    cohort_summary = {}
    
    # Best model
    best_model_file = CALCULATOR_DIR / "outputs" / "models" / cohort / "best_model.txt"
    if best_model_file.exists():
        with open(best_model_file, 'r') as f:
            content = f.read()
            lines = content.split('\n')
            for line in lines:
                if line.startswith("Best Model:"):
                    cohort_summary["best_model"] = line.replace("Best Model: ", "").strip()
                elif line.startswith("C-index:"):
                    try:
                        cohort_summary["c_index"] = float(line.replace("C-index: ", "").strip())
                    except:
                        pass
    
    # Dashboard data
    dashboard_file = CALCULATOR_DIR / "outputs" / "shap_ffa" / cohort / "dashboard_data.json"
    if dashboard_file.exists():
        with open(dashboard_file, 'r') as f:
            dashboard_data = json.load(f)
            cohort_summary["top_factors_count"] = len(dashboard_data.get('top_causal_factors', []))
            if dashboard_data.get('top_causal_factors'):
                cohort_summary["top_factor"] = dashboard_data['top_causal_factors'][0]['feature']
                cohort_summary["top_factor_importance"] = dashboard_data['top_causal_factors'][0].get(
                    'causal_responsibility', 
                    dashboard_data['top_causal_factors'][0].get('importance', 0)
                )
    
    summary["cohorts"][cohort] = cohort_summary

# Save summary
summary_file = CALCULATOR_DIR / "outputs" / "workflow_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Workflow summary saved to: {summary_file}")
print("\nSummary:")
print(json.dumps(summary, indent=2))

## 7. Quick Run: All Cohorts

Run the complete workflow for all three cohorts.

In [ ]:
# Run complete workflow for all cohorts
ALL_COHORTS = ["Combined", "CHD", "Myocardio"]

print(f"\n{'=' * 80}")
print("Running Complete Workflow for All Cohorts")
print(f"{'=' * 80}")

# Train all models
print("\nStep 1: Training all models...")
for cohort in ALL_COHORTS:
    print(f"\n  Training {cohort}...")
    try:
        train_models_for_cohort(cohort)
        print(f"  ✓ {cohort} complete")
    except Exception as e:
        print(f"  ✗ {cohort} failed: {e}")

# Run SHAP/FFA for all cohorts
print("\n\nStep 2: Running SHAP/FFA analysis...")
for cohort in ALL_COHORTS:
    print(f"\n  Analyzing {cohort}...")
    try:
        result = subprocess.run(
            [
                sys.executable,
                str(CALCULATOR_DIR / "run_shap_ffa_workflow.py"),
                "--cohort", cohort,
                "--top-k", str(TOP_K)
            ],
            cwd=str(CALCULATOR_DIR),
            capture_output=False,
            text=True
        )
        if result.returncode == 0:
            print(f"  ✓ {cohort} complete")
        else:
            print(f"  ⚠ {cohort} exited with code: {result.returncode}")
    except Exception as e:
        print(f"  ✗ {cohort} failed: {e}")

print(f"\n{'=' * 80}")
print("Complete workflow finished!")
print(f"{'=' * 80}")